# 12.11 Community detection

Communities have more internal connection than the graph expects by chance. Community detection uses adjacency, degree, and spectral structure to find unsupervised graph groups that are denser than a degree-preserving null model. This walkthrough keeps the same modularity method from a four-node graph through larger noisy graphs. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 1215
random.seed(SEED)
np.random.seed(SEED)


def make_d1_graph():
    graph = nx.Graph()
    graph.add_nodes_from(range(4))
    graph.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3)])
    labels = {0: 0, 1: 0, 2: 1, 3: 1}
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_karate_rung():
    graph = nx.karate_club_graph()
    labels = {}
    for node, data in graph.nodes(data=True):
        labels[node] = 0 if data.get("club") == "Mr. Hi" else 1
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_sbm_rung(sizes, p_in, p_out, seed, noise_edges=0):
    probs = []
    for row in range(len(sizes)):
        prob_row = []
        for col in range(len(sizes)):
            prob_row.append(p_in if row == col else p_out)
        probs.append(prob_row)
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    for _ in range(noise_edges):
        u = int(rng.choice(nodes))
        v = int(rng.choice(nodes))
        if u != v:
            graph.add_edge(u, v)
    labels = {}
    offset = 0
    for block, size in enumerate(sizes):
        for node in range(offset, offset + size):
            labels[node] = block
        offset = offset + size
    positions = nx.spring_layout(graph, seed=seed)
    return graph, labels, positions


def make_cora_like_rung():
    graph, labels, positions = make_sbm_rung([18, 16, 14], 0.26, 0.035, SEED + 4, noise_edges=8)
    rng = np.random.default_rng(SEED + 44)
    years = {}
    node_types = {}
    for node in graph.nodes():
        years[node] = int(2016 + rng.integers(0, 7))
        node_types[node] = "paper"
    nx.set_node_attributes(graph, years, "year")
    nx.set_node_attributes(graph, node_types, "kind")
    return graph, labels, positions


def make_large_noisy_rung():
    graph, labels, positions = make_sbm_rung([35, 32, 30, 28], 0.18, 0.045, SEED + 5, noise_edges=55)
    rng = np.random.default_rng(SEED + 55)
    for u, v in graph.edges():
        graph[u][v]["time"] = int(rng.integers(0, 10))
        graph[u][v]["relation"] = "strong" if labels[u] == labels[v] else "weak"
    return graph, labels, positions


def build_graph_ladder():
    d1_graph, d1_labels, d1_pos = make_d1_graph()
    d2_graph, d2_labels, d2_pos = make_karate_rung()
    d3_graph, d3_labels, d3_pos = make_sbm_rung([12, 12, 12], 0.32, 0.04, SEED + 3, noise_edges=5)
    d4_graph, d4_labels, d4_pos = make_cora_like_rung()
    d5_graph, d5_labels, d5_pos = make_large_noisy_rung()
    return [
        {"name": "D1 toy", "graph": d1_graph, "labels": d1_labels, "pos": d1_pos},
        {"name": "D2 karate", "graph": d2_graph, "labels": d2_labels, "pos": d2_pos},
        {"name": "D3 SBM", "graph": d3_graph, "labels": d3_labels, "pos": d3_pos},
        {"name": "D4 Cora-like subset", "graph": d4_graph, "labels": d4_labels, "pos": d4_pos},
        {"name": "D5 larger noisy", "graph": d5_graph, "labels": d5_labels, "pos": d5_pos},
    ]


def partition_from_labels(labels):
    groups = defaultdict(set)
    for node, label in labels.items():
        groups[label].add(node)
    return list(groups.values())


def accuracy_score(y_true, y_pred):
    correct = 0
    total = 0
    for node, truth in y_true.items():
        if node in y_pred:
            correct = correct + int(y_pred[node] == truth)
            total = total + 1
    return correct / max(total, 1)


## The concept, built once (D1): modularity and conductance

Modularity compares observed within-community edges to the degree-preserving null model:

$$Q=\frac{1}{2m}\sum_{ij}\(A_{ij}-d_i d_j/(2m))\mathbf{1}[c_i=c_j].$$

On the lesson graph, partition $\{0,1,2\}|\{3\}$ must give $Q=-0.03125$ and $S=\{0,1\}$ has conductance $2/4=0.500$.

In [ ]:

def modularity_value(graph, communities, resolution=1.0):
    return nx.algorithms.community.quality.modularity(
        graph,
        communities,
        resolution=resolution,
    )


def conductance_value(graph, subset):
    subset = set(subset)
    cut = nx.cut_size(graph, subset)
    volume = sum(dict(graph.degree(subset)).values())
    complement = set(graph.nodes()) - subset
    other_volume = sum(dict(graph.degree(complement)).values())
    return cut / min(volume, other_volume)


## Add spectral and label-propagation views

A real community workflow triangulates modularity with spectral signs and label propagation because each one can fail differently.

In [ ]:

def spectral_split(graph):
    laplacian = nx.normalized_laplacian_matrix(graph).toarray()
    values, vectors = np.linalg.eigh(laplacian)
    fiedler = vectors[:, 1]
    nodes = list(graph.nodes())
    positive = {nodes[index] for index, value in enumerate(fiedler) if value >= 0}
    negative = set(nodes) - positive
    if len(positive) == 0 or len(negative) == 0:
        median = float(np.median(fiedler))
        positive = {nodes[index] for index, value in enumerate(fiedler) if value >= median}
        negative = set(nodes) - positive
    return [positive, negative]


def one_label_propagation_step(graph, labels):
    new_labels = {}
    for node in sorted(graph.nodes()):
        votes = [labels[node]]
        for neighbor in graph.neighbors(node):
            votes.append(labels[neighbor])
        counts = Counter(votes)
        best_count = max(counts.values())
        best_labels = [label for label, count in counts.items() if count == best_count]
        new_labels[node] = min(best_labels)
    return [new_labels[node] for node in sorted(graph.nodes())]


def detect_communities(graph, resolution=1.0):
    communities = list(nx.algorithms.community.greedy_modularity_communities(
        graph,
        resolution=resolution,
    ))
    modularity = modularity_value(graph, communities, resolution=resolution)
    split = spectral_split(graph)
    return communities, modularity, split


graph, labels, positions = make_d1_graph()
partition_a = [{0, 1, 2}, {3}]
partition_b = [{0, 1}, {2, 3}]
q_a = modularity_value(graph, partition_a)
q_b = modularity_value(graph, partition_b)
step_labels = one_label_propagation_step(graph, {0: 0, 1: 0, 2: 1, 3: 1})
cond = conductance_value(graph, {0, 1})
split = spectral_split(graph)
print("Q({0,1,2}|{3}) =", round(q_a, 5))
print("Q({0,1}|{2,3}) =", round(q_b, 5))
print("label propagation step =", step_labels)
print("conductance({0,1}) =", round(cond, 3))
assert abs(q_a - (-0.03125)) < 1e-12
assert q_b > q_a
assert step_labels == [0, 0, 0, 1]
assert abs(cond - 0.500) < 1e-12
assert len(split[0]) > 0
assert len(split[1]) > 0


## The dataset ladder

D1 is the four-node lesson graph; D2 is karate club; D3 is a noisy SBM; D4 is a Cora-like citation subset; D5 is a larger noisy graph.

In [ ]:

ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    labels = rung["labels"]
    sample_nodes = list(graph.nodes())[:5]
    sample_edges = list(graph.edges())[:5]
    print(rung["name"])
    print("  nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "classes", sorted(set(labels.values())))
    print("  sample nodes", sample_nodes)
    print("  sample edges", sample_edges)


## Run the same method across D1-D5

In [ ]:

ladder = build_graph_ladder()
community_results = []
for rung in ladder:
    graph = rung["graph"]
    communities, score, split = detect_communities(graph)
    rung["communities"] = communities
    rung["community_score"] = score
    community_results.append(score)
    print(f"{rung['name']}: nodes={graph.number_of_nodes()} edges={graph.number_of_edges()} communities={len(communities)} modularity={score:.3f}")


## Results visualization

Top row: graph panels colored by detected communities. Bottom left: modularity curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    positions = rung["pos"]
    communities = rung["communities"]
    color_map = {}
    for group_index, group in enumerate(communities):
        for node in group:
            color_map[node] = group_index
    node_colors = [color_map.get(node, 0) for node in graph.nodes()]
    nx.draw_networkx(
        graph,
        pos=positions,
        node_color=node_colors,
        cmap="tab10",
        with_labels=False,
        node_size=55,
        edge_color="#cccccc",
        ax=axes[0, index],
    )
    axes[0, index].set_title(rung["name"])
    axes[0, index].axis("off")
axes[1, 0].plot(range(1, 6), community_results, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("modularity")
axes[1, 0].set_title("modularity vs complexity")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung: resolution limit

The $2m$ denominator can merge small real communities inside a larger graph. Reproduce it with a ring of cliques, then report a multiscale resolution sweep.

In [ ]:

ring = nx.ring_of_cliques(14, 5)
coarse = list(nx.algorithms.community.greedy_modularity_communities(ring, resolution=1.0))
fine = list(nx.algorithms.community.greedy_modularity_communities(ring, resolution=2.0))
coarse_sizes = sorted([len(group) for group in coarse])
fine_sizes = sorted([len(group) for group in fine])
coarse_q = modularity_value(ring, coarse, resolution=1.0)
fine_q = modularity_value(ring, fine, resolution=2.0)
print("resolution=1 groups", len(coarse), coarse_sizes[:6], "Q", round(coarse_q, 3))
print("resolution=2 groups", len(fine), fine_sizes[:6], "Q", round(fine_q, 3))
print("Fix: report multiscale partitions instead of trusting one modularity denominator.")



## Evaluate it + practice

- Metric: compare the rung's modularity against the no-skill baseline `one community or random partition` on D1-D5.
- Sanity check: rerun with the fixed seed and verify D1 reproduces the asserted lesson numbers before trusting larger rungs.
- Ablation: turn off the resolution sweep and trust a single greedy modularity partition; the metric should drop or the failure should become visible.
- Failure signal: a high score with a violated split, symmetry, or relation contract is not a real win.
- Inspect the hardest rung by plotting both the graph artifact and the metric curve rather than reading one scalar alone.

Practice prompts:
1. Change only the D3 noise level and predict how the metric curve should move.


2. Replace the D5 seed with another fixed seed and compare the artifact panel.

3. Add one cheap assertion that would catch the lesson pitfall before the metric is printed.